# Production Architecture: Real-Time Arabic (ArSL) Sign Language Recognition

### What changed from baseline
- **`mp.solutions.hands` only, `max_num_hands=1`** — face/pose detection removed entirely
- **Threaded camera** with drop-queue — always the latest frame, no buffer lag
- **`@tf.function` compiled inference** — graph-mode, no Python overhead per prediction
- **`model(x, training=False)`** — bypasses `predict()` batch pipeline
- **Raised MediaPipe thresholds** (`detection=0.70`, `tracking=0.65`)
- **Stabilisation params kept at 15/11/0.75/1.2** — production-tuned values
- **Arabic text rendering** via PIL + `arabic_reshaper` + `python-bidi` (unchanged)


## 0. Imports

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import time
import threading
import queue
from collections import deque
import os

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 1. GPU Configuration

In [2]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU configured: {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"⚠ GPU config error: {e}")
else:
    print("ℹ No GPU detected, using CPU")

✓ GPU configured: 1 GPU(s)


## 2. Configuration & Parameters

31 classes: 28 Arabic letters + space + del + nothing.
All stabilisation values kept at production-tuned settings.


In [3]:
MODEL_PATH = r"MediaPipe_ARLetters_training"

STABILIZATION_WINDOW_SIZE  = 15
STABILIZATION_THRESHOLD    = 11     # 11/15 ≈ 73%
MIN_CONFIDENCE             = 0.75
HOLD_COOLDOWN_SECONDS      = 1.2

MP_DETECTION_CONFIDENCE    = 0.70   # raised from 0.5
MP_TRACKING_CONFIDENCE     = 0.65   # raised from 0.5

DISPLAY_WIDTH  = 1920
DISPLAY_HEIGHT = 1080

CLASS_LABELS = [
    "ain",    # 0  → ع
    "al",     # 1  → ال
    "aleff",  # 2  → ا
    "bb",     # 3  → ب
    "dal",    # 4  → د
    "dha",    # 5  → ذ
    "dhad",   # 6  → ض
    "fa",     # 7  → ف
    "gaaf",   # 8  → ق
    "ghain",  # 9  → غ
    "ha",     # 10 → ه
    "haa",    # 11 → ح
    "jeem",   # 12 → ج
    "kaaf",   # 13 → ك
    "khaa",   # 14 → خ
    "la",     # 15 → لا
    "laam",   # 16 → ل
    "meem",   # 17 → م
    "nun",    # 18 → ن
    "ra",     # 19 → ر
    "saad",   # 20 → ص
    "seen",   # 21 → س
    "sheen",  # 22 → ش
    "ta",     # 23 → ة
    "taa",    # 24 → ت
    "thaa",   # 25 → ث
    "thal",   # 26 → ذ (variant)
    "toot",   # 27 → ط
    "waw",    # 28 → و
    "ya",     # 29 → ي
    "yaa",    # 30 → ي (variant)
    "zay",    # 31 → ز
]

ARABIC_MAP = {
    "ain"   : "ع", "al"    : "ال", "aleff" : "ا",
    "bb"    : "ب", "dal"   : "د",  "dha"   : "ذ",
    "dhad"  : "ض", "fa"    : "ف",  "gaaf"  : "ق",
    "ghain" : "غ", "ha"    : "ه",  "haa"   : "ح",
    "jeem"  : "ج", "kaaf"  : "ك",  "khaa"  : "خ",
    "la"    : "لا","laam"  : "ل",  "meem"  : "م",
    "nun"   : "ن", "ra"    : "ر",  "saad"  : "ص",
    "seen"  : "س", "sheen" : "ش",  "ta"    : "ة",
    "taa"   : "ت", "thaa"  : "ث",  "thal"  : "ذ",
    "toot"  : "ط", "waw"   : "و",  "ya"    : "ي",
    "yaa"   : "ي", "zay"   : "ز",
}

print(f"✓ Settings configured — {len(CLASS_LABELS)} classes, ARABIC_MAP ready")



✓ Settings configured — 32 classes, ARABIC_MAP ready


## 3. Threaded Camera

In [4]:

class ThreadedCamera:
    """Background daemon thread keeps queue fresh (size 1 = always latest frame)."""
    def __init__(self, src=0):
        self.cap = cv2.VideoCapture(src, cv2.CAP_DSHOW)
        self.cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M','J','P','G'))
        self.cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH,  DISPLAY_WIDTH)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, DISPLAY_HEIGHT)
        self.cap.set(cv2.CAP_PROP_FPS, 30)
        self._q      = queue.Queue(maxsize=1)
        self._active = True
        self._thread = threading.Thread(target=self._reader, daemon=True)
        self._thread.start()

    def _reader(self):
        while self._active:
            ret, frame = self.cap.read()
            if not ret:
                continue
            if not self._q.empty():
                try:
                    self._q.get_nowait()
                except queue.Empty:
                    pass
            self._q.put(frame)

    def read(self):
        return self._q.get()

    def is_opened(self):
        return self.cap.isOpened()

    def release(self):
        self._active = False
        self._thread.join(timeout=2)
        self.cap.release()

print("✓ ThreadedCamera defined")

✓ ThreadedCamera defined


## 4. MediaPipe Hands (single hand)

`max_num_hands=1` — no face/pose. 21 pts × 3 coords = 63 features.


In [5]:
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode        = False,
    max_num_hands            = 1,
    min_detection_confidence = MP_DETECTION_CONFIDENCE,
    min_tracking_confidence  = MP_TRACKING_CONFIDENCE,
)

def extract_features(results):
    """Returns (1, 63) float32 or None. First detected hand only."""
    if not results.multi_hand_landmarks:
        return None
    lm = results.multi_hand_landmarks[0].landmark
    return np.array([[p.x, p.y, p.z] for p in lm], dtype=np.float32).reshape(1, -1)

print("✓ MediaPipe Hands configured (max_num_hands=1)")

✓ MediaPipe Hands configured (max_num_hands=1)


## 5. Stabilisation Tracker

Same logic as English. Includes the unlock fix for different-sign-during-cooldown.


In [6]:
class StabilizationTracker:
    def __init__(self):
        self.buffer          = deque(maxlen=STABILIZATION_WINDOW_SIZE)
        self.committed_label = None
        self.cooldown_until  = 0.0

    def update(self, label, confidence):
        now = time.time()
        if label is None or label == "nothing":
            self.buffer.clear()
            self.committed_label = None
            return None, 0.0, "waiting", 0
        if self.committed_label == label and now < self.cooldown_until:
            return label, confidence, "cooldown", 100
        if self.committed_label is not None and self.committed_label != label:
            self.committed_label = None
        self.buffer.append(label)
        count    = self.buffer.count(label)
        progress = int(min(100, (count / STABILIZATION_THRESHOLD) * 100))
        if count >= STABILIZATION_THRESHOLD and len(self.buffer) == STABILIZATION_WINDOW_SIZE:
            self.committed_label = label
            self.cooldown_until  = now + HOLD_COOLDOWN_SECONDS
            self.buffer.clear()
            return label, confidence, "committed", 100
        return label, confidence, "stabilizing", progress

print("✓ StabilizationTracker defined")

✓ StabilizationTracker defined


## 6. Load Model & Compile Inference

The Arabic MLP was trained with `mixed_float16`. It is cloned under `float32` so it
runs correctly on both GPU and CPU. `@tf.function` compiles the graph for fast inference.


In [7]:
#model = "arsl_mediapipe_mlp_model_bestV2.2"
MODEL_PATH="arsl_mediapipe_mlp_model_bestV2.2.h5"
try:
    if os.path.exists(MODEL_PATH):
        model_raw = tf.keras.models.load_model(MODEL_PATH)
        # Clone under float32 so it runs on both GPU and CPU without precision errors
        model = tf.keras.models.clone_model(model_raw)
        model.set_weights(model_raw.get_weights())
        del model_raw
        print("✓ Arabic MLP model loaded (cast to float32)")
    else:
        print(f"❌ Model not found at {MODEL_PATH}")
except Exception as e:
    print(f"❌ Error loading model: {e}")

if model is not None:
    @tf.function(input_signature=[tf.TensorSpec(shape=[1, 63], dtype=tf.float32)])
    def run_inference(x):
        return model(x, training=False)

    _ = run_inference(tf.zeros([1, 63], dtype=tf.float32))  # warm-up
    print("✓ Inference compiled and warmed up")

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6
✓ Arabic MLP model loaded (cast to float32)
✓ Inference compiled and warmed up


## 7. Arabic Text Rendering Utility

Arabic letters change shape by position, and the script is right-to-left.
`arabic_reshaper` fixes shaping; `python-bidi` fixes direction.
Falls back to plain OpenCV text if these libraries are absent.


In [8]:
try:
    from PIL import Image, ImageDraw, ImageFont
    import arabic_reshaper
    from bidi.algorithm import get_display
    ARABIC_TEXT_OK = True
    print("✓ Arabic text rendering libraries found")
except ImportError:
    ARABIC_TEXT_OK = False
    print("⚠ Arabic text libraries missing. Run: pip install arabic-reshaper python-bidi pillow")

# Font search order: try common system paths first
_FONT_CANDIDATES = [
    "arial.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/System/Library/Fonts/Arial.ttf",
    "C:/Windows/Fonts/arial.ttf",
]

def _get_font(size):
    for path in _FONT_CANDIDATES:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()

def draw_arabic_text(frame, text, position, font_size=40, color=(255, 255, 255)):
    """Render Arabic text correctly on an OpenCV frame."""
    if not text:
        return frame
    if not ARABIC_TEXT_OK:
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        return frame
    reshaped  = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)
    pil_img   = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw      = ImageDraw.Draw(pil_img)
    font      = _get_font(font_size)
    rgb_color = (color[2], color[1], color[0])
    draw.text(position, bidi_text, fill=rgb_color, font=font)
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

print("✓ draw_arabic_text defined")

✓ Arabic text rendering libraries found
✓ draw_arabic_text defined


## 8. Main Recognition Loop

In [9]:
def run_sign_recognition():
    if model is None:
        print("❌ Model not loaded.")
        return
    
    cam = ThreadedCamera(src=0)
    if not cam.is_opened():
        print("❌ Cannot access camera")
        return

    print("\n================================================")
    print("🤟 ARABIC SIGN LANGUAGE RECOGNITION — PRODUCTION")
    print("================================================")
    print("  q = quit    c = clear sentence")
    print("================================================\n")

    window_name = "Arabic SLR — Production"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, DISPLAY_WIDTH, DISPLAY_HEIGHT)

    tracker            = StabilizationTracker()
    predicted_sentence = ""
    fps_start          = time.time()
    frame_count        = 0
    fps_display        = 0

    try:
        while True:
            frame = cam.read()
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results = hands.process(rgb)
            rgb.flags.writeable = True

            display_status = "No hand detected"
            status_color   = (150, 150, 150)

            # Check if landmarks are detected before proceeding
            if results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame, results.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS)
                
                # FIXED: features and inference are now properly nested
                features = extract_features(results) 
                
                if features is not None:
                    try:
                        prediction = run_inference(tf.constant(features))[0].numpy()
                        class_idx  = int(np.argmax(prediction))
                        conf       = float(prediction[class_idx])
                        raw_label  = CLASS_LABELS[class_idx] if class_idx < len(CLASS_LABELS) else "nothing"

                        if conf < MIN_CONFIDENCE:
                            display_status = f"Low confidence: {conf:.0%}"
                            status_color   = (0, 100, 255)
                            tracker.update(None, 0.0)
                        else:
                            tracked_label, tracked_conf, status, progress = tracker.update(raw_label, conf)
                            tracked_arabic = ARABIC_MAP.get(tracked_label, tracked_label) if tracked_label else ""
                            
                            if status == "stabilizing":
                                display_status = f"{tracked_arabic} ({tracked_conf:.0%})  {progress}%"
                                status_color   = (0, 255, 255)
                            elif status == "cooldown":
                                display_status = f"{tracked_arabic} ({tracked_conf:.0%})  ✓ Committed"
                                status_color   = (255, 200, 0)
                            elif status == "committed":
                                display_status = f"{tracked_arabic} ({tracked_conf:.0%})  ✓ COMMITTED!"
                                status_color   = (0, 255, 0)
                                if tracked_label == "space":
                                    if not predicted_sentence.endswith(" "):
                                        predicted_sentence += " "
                                elif tracked_label == "del":
                                    predicted_sentence = predicted_sentence[:-1]
                                elif tracked_label not in ("nothing", "label"):
                                    predicted_sentence += tracked_arabic

                    except Exception as e:
                        print(f"⚠ Inference error: {e}")
                        tracker.update(None, 0.0)
                else:
                    tracker.update(None, 0.0)
            else:
                tracker.update(None, 0.0)

            # Render UI
            frame = cv2.flip(frame, 1)
            frame_count += 1
            if time.time() - fps_start >= 1.0:
                fps_display = frame_count
                frame_count = 0
                fps_start   = time.time()

            # HUD — top bar
            cv2.rectangle(frame, (0, 0), (DISPLAY_WIDTH, 70), (30, 30, 30), -1)
            cv2.putText(frame, f"FPS: {fps_display}  |  q=quit  c=clear",
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

            # Status line (Arabic support)
            frame = draw_arabic_text(frame, display_status, (10, 42), font_size=28, color=status_color)

            # Sentence bar — bottom
            bar_h = 80
            cv2.rectangle(frame,
                          (0, DISPLAY_HEIGHT - bar_h), (DISPLAY_WIDTH, DISPLAY_HEIGHT),
                          (30, 30, 30), -1)
            
            frame = draw_arabic_text(
                frame,
                predicted_sentence[-40:] if predicted_sentence else "_",
                (20, DISPLAY_HEIGHT - 65), font_size=44, color=(255, 255, 255))

            cv2.imshow(window_name, frame)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("c"):
                predicted_sentence = ""
                tracker.update(None, 0.0)
                print("🗑 Sentence cleared")
                
    finally:
        cam.release()
        cv2.destroyAllWindows()
        print(f"\n📝 Final sentence: {predicted_sentence}")

# Start the application
run_sign_recognition()


🤟 ARABIC SIGN LANGUAGE RECOGNITION — PRODUCTION
  q = quit    c = clear sentence


📝 Final sentence: ضض


In [10]:
import tensorflow as tf, os

# Check which model files actually exist
for f in os.listdir("."):
    if f.endswith(".h5"):
        print(f)

# Check model output shape
m = tf.keras.models.load_model("arsl_mediapipe_mlp_model_bestV2.2.h5")
print("Output units:", m.output_shape)   # should be (None, 35) for 35 classes


arsl_mediapipe_mlp_model_bestV1.h5
arsl_mediapipe_mlp_model_bestV2.2.h5
mobilenet_arabic_final.h5
Output units: (None, 32)
